In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.Defaulting to user installation because normal site-packages is not writeable



You should consider upgrading via the 'c:\Program Files (x86)\Microsoft Visual Studio\Shared\Python39_64\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_base_copro.csv')

df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()


df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code', 'dept_nom']).agg({
    'lots_habitation': 'sum',
    'lots_parking': 'sum',
    'lat': 'first',
    'long': 'first'
}).reset_index()


df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)
df_final = df_unique[df_unique['score_immeuble'] <= 5].copy()

df_final['code_postal'] = df_final['code_postal'].fillna(0).astype(int).astype(str)



kpi1_score_ville = df_final.groupby(['code_postal', 'ville'])['score_immeuble'].mean().reset_index()

kpi1_score_ville = kpi1_score_ville.rename(columns={'score_immeuble': 'score_potentiel_ville'})
kpi1_score_ville['score_potentiel_ville'] = kpi1_score_ville['score_potentiel_ville'].round(3)

kpi1_score_ville = kpi1_score_ville.sort_values('score_potentiel_ville', ascending=False)

print("KPI 1 - Score par ville :")
display(kpi1_score_ville.head(10))

KPI 1 - Score par ville :


,code_postal,ville,score_potentiel_ville
4137,38530,La Buissière,2.3333
4282,38960,SAINT-AUPRE,2.3333
11700,85480,THORIGNY,2.3333
8772,69380,MARCILLY D AZERGUES,2.3333
279,13010,MARSEILLE 16,2.3125
567,13990,FONTVIEILLE,2.3077
7823,67150,MATZENHEIM,2.2917
12128,9130,CARLA BAYLE,2.2833
7889,67230,HUTTENHEIM,2.2500
1218,20233,PIETRACORBARA,2.2500
